# Đồ thị Hỗ trợ/Xung đột + Chọn top 3 tổ hợp thuốc (Google Colab)

Quy trình:
1. Xây 2 đồ thị Neo4j (`HO_TRO`, `XUNG_DOT`) từ 2 file CSV.
2. Train PyKEEN (Knowledge Graph Embedding).
3. Nhập danh sách viên thuốc: **chắc chắn** = 1 tên, **nghi ngờ** = 2–3 tên cách nhau bằng `/`.
4. Thử mọi kịch bản, **loại tổ hợp có cặp xung đột đã khai báo**, **cộng điểm thưởng cho cặp hỗ trợ đã khai báo**, in **top 3** tổ hợp điểm cao nhất.

**Trước khi chạy:**
1. Tạo Neo4j Aura Free tại https://neo4j.com/cloud/aura/ → lấy `URI`, `username`, `password`.
2. Upload 2 file `hotrodatach.csv`, `Workbook2.csv` lên Google Drive (ví dụ thư mục `NCKH/`).

## 1. Cài đặt thư viện

In [7]:
!pip install -q neo4j pykeen pandas

## 2. Kết nối Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Cấu hình

In [9]:
import os

# ── Đường dẫn trên Google Drive ──
FILE_HO_TRO    = "/content/drive/MyDrive/NCKH/support.csv"
FILE_XUNG_DOT  = "/content/drive/MyDrive/NCKH/conflict.csv"

# ── Thông tin Neo4j Aura ──
NEO4J_URI      = "neo4j+s://711f4fec.databases.neo4j.io"  # đổi thành URI thật
NEO4J_USER     = "711f4fec"
NEO4J_PASSWORD = "kQ62GGuRi_2X5DEMNPNf6XOfIcXvga0ihBl266HcHm0"              # đổi thành password thật

# ── PyKEEN ──
PYKEEN_MODEL   = "TransE"
PYKEEN_EPOCHS  = 100

# ── Chấm điểm ──
DIEM_THUONG_HO_TRO = 0.5   # điểm cộng thêm cho mỗi cặp đã khai báo HỖ TRỢ
TOP_K              = 3     # số tổ hợp tốt nhất cần in
TOI_DA_KICH_BAN    = 5000  # giới hạn số kịch bản (tránh bùng nổ tổ hợp)

for ten, path in [("FILE_HO_TRO", FILE_HO_TRO), ("FILE_XUNG_DOT", FILE_XUNG_DOT)]:
    print(("[OK] " if os.path.exists(path) else "[CẢNH BÁO] Không tìm thấy ") + f"{ten}: {path}")

[OK] FILE_HO_TRO: /content/drive/MyDrive/NCKH/support.csv
[OK] FILE_XUNG_DOT: /content/drive/MyDrive/NCKH/conflict.csv


## 4. Hàm xây đồ thị Neo4j + train PyKEEN

In [10]:
import re
from itertools import combinations, product

import numpy as np
import pandas as pd
import torch
from neo4j import GraphDatabase
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory


def _xoa_du_lieu_cu(session):
    session.run("MATCH (n:Thuoc) DETACH DELETE n")


def _create_indexes(session):
    session.run("CREATE INDEX thuoc_ten IF NOT EXISTS FOR (t:Thuoc) ON (t.ten)")


def _import_ho_tro(session, df):
    rows, seen = [], set()
    for _, row in df.iterrows():
        a, b = str(row["Tên thuốc"]).strip(), str(row["Thuốc hỗ trợ"]).strip()
        if (a, b) not in seen and (b, a) not in seen:
            seen.add((a, b))
            rows.append({"ten_thuoc": a, "thuoc_ho_tro": b,
                         "cach_thuc": str(row["Cách thức hỗ trợ cụ thể"]).strip()})
    session.run("""
        UNWIND $rows AS row
        MERGE (a:Thuoc {ten: row.ten_thuoc})
        MERGE (b:Thuoc {ten: row.thuoc_ho_tro})
        MERGE (a)-[r:HO_TRO]->(b)
          ON CREATE SET r.cach_thuc = row.cach_thuc
          ON MATCH  SET r.cach_thuc = row.cach_thuc
    """, rows=rows)


def _import_xung_dot(session, df):
    rows, seen = [], set()
    for _, row in df.iterrows():
        a, b = str(row["Thuốc chính"]).strip(), str(row["Thuốc xung đột"]).strip()
        if (a, b) not in seen and (b, a) not in seen:
            seen.add((a, b))
            rows.append({"thuoc_chinh": a, "thuoc_xung_dot": b,
                         "ly_do": str(row["Lý do xung đột"]).strip()})
    session.run("""
        UNWIND $rows AS row
        MERGE (a:Thuoc {ten: row.thuoc_chinh})
        MERGE (b:Thuoc {ten: row.thuoc_xung_dot})
        MERGE (a)-[r:XUNG_DOT]->(b)
          ON CREATE SET r.ly_do = row.ly_do
          ON MATCH  SET r.ly_do = row.ly_do
    """, rows=rows)


def _lay_triple(session):
    rows_ht = session.run("MATCH (a:Thuoc)-[:HO_TRO]->(b:Thuoc) RETURN a.ten AS h, 'HO_TRO' AS r, b.ten AS t").data()
    rows_xd = session.run("MATCH (a:Thuoc)-[:XUNG_DOT]->(b:Thuoc) RETURN a.ten AS h, 'XUNG_DOT' AS r, b.ten AS t").data()
    return [(d['h'], d['r'], d['t']) for d in rows_ht + rows_xd]


def _lay_xung_dot(session):
    rows = session.run("MATCH (a:Thuoc)-[r:XUNG_DOT]->(b:Thuoc) RETURN a.ten AS a, b.ten AS b, r.ly_do AS ly_do").data()
    return {frozenset({r['a'], r['b']}): r['ly_do'] for r in rows}


def _lay_ho_tro(session):
    rows = session.run("MATCH (a:Thuoc)-[r:HO_TRO]->(b:Thuoc) RETURN a.ten AS a, b.ten AS b, r.cach_thuc AS cach_thuc").data()
    return {frozenset({r['a'], r['b']}): r['cach_thuc'] for r in rows}


def xay_do_thi_va_lay_du_lieu():
    print("=" * 60)
    print("GIAI ĐOẠN 1: XÂY ĐỒ THỊ HỖ TRỢ + XUNG ĐỘT TRONG NEO4J")
    print("=" * 60)

    df_ho_tro   = pd.read_csv(FILE_HO_TRO)
    df_xung_dot = pd.read_csv(FILE_XUNG_DOT)

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        _xoa_du_lieu_cu(session)
        _create_indexes(session)
        _import_ho_tro(session, df_ho_tro)
        _import_xung_dot(session, df_xung_dot)

        cap_xung_dot = _lay_xung_dot(session)
        cap_ho_tro   = _lay_ho_tro(session)
        all_triples  = _lay_triple(session)
    driver.close()

    print(f"  Đã nạp {len(cap_ho_tro)} cặp hỗ trợ, {len(cap_xung_dot)} cặp xung đột.")
    print(f"  Tổng {len(all_triples)} triple để train PyKEEN.\n")
    return cap_xung_dot, cap_ho_tro, all_triples


def train_pykeen(triples):
    print("=" * 60)
    print("GIAI ĐOẠN 2: TRAIN PYKEEN (Knowledge Graph Embedding)")
    print("=" * 60)

    MIN_TRIPLE = 3
    if len(triples) < MIN_TRIPLE:
        print(f"  [CẢNH BÁO] Chỉ có {len(triples)} triple, cần ≥ {MIN_TRIPLE}. Bỏ qua PyKEEN.")
        return None, None

    triples_np = np.array(triples, dtype=str)
    tf_full = TriplesFactory.from_labeled_triples(triples_np)

    if len(triples) >= 10:
        train_tf, test_tf, valid_tf = tf_full.split([0.8, 0.1, 0.1], random_state=42)
        result = pipeline(training=train_tf, testing=test_tf, validation=valid_tf,
                          model=PYKEEN_MODEL, training_kwargs=dict(num_epochs=PYKEEN_EPOCHS),
                          random_seed=42)
    else:
        train_tf, test_tf = tf_full.split([0.8, 0.2], random_state=42)
        result = pipeline(training=train_tf, testing=test_tf,
                          model=PYKEEN_MODEL, training_kwargs=dict(num_epochs=PYKEEN_EPOCHS),
                          random_seed=42)

    print(f"  Train xong! Loss cuối: {result.losses[-1]:.4f}\n")
    return result.model, tf_full


def _tinh_score(model, tf, quan_he, thuoc_a, thuoc_b):
    if model is None or tf is None:
        return 0.0
    try:
        e2id, r2id = tf.entity_to_id, tf.relation_to_id
        if thuoc_a not in e2id or thuoc_b not in e2id or quan_he not in r2id:
            return 0.0
        h = torch.tensor([e2id[thuoc_a]])
        r = torch.tensor([r2id[quan_he]])
        t = torch.tensor([e2id[thuoc_b]])
        with torch.no_grad():
            score = model.score_hrt(torch.stack([h, r, t], dim=1))
        return float(score.item())
    except Exception:
        return 0.0


def tinh_score_ho_tro(model, tf, a, b):
    return _tinh_score(model, tf, "HO_TRO", a, b)


def tinh_score_xung_dot(model, tf, a, b):
    return _tinh_score(model, tf, "XUNG_DOT", a, b)


print("Đã nạp xong các hàm xây đồ thị + train.")

Đã nạp xong các hàm xây đồ thị + train.


## 5. Hàm chấm điểm tổ hợp (loại xung đột, thưởng hỗ trợ, top K)

In [11]:
def tinh_diem_cap_cache(a, b, cache, model, tf):
    """Điểm PyKEEN của 1 cặp thuốc (có cache)."""
    key = frozenset({a, b})
    if key not in cache:
        ht = tinh_score_ho_tro(model, tf, a, b)
        xd = tinh_score_xung_dot(model, tf, a, b)
        cache[key] = (ht, xd, ht - xd)
    return cache[key]


def tinh_diem_kich_ban(kich_ban, cache, cap_xung_dot, cap_ho_tro, model, tf):
    """
    Điểm tổng của 1 kịch bản (mỗi viên đã chốt 1 tên thuốc).
    Trả về (None, lý do) nếu có cặp xung đột đã khai báo -> loại luôn.
    """
    tong, chi_tiet = 0.0, []
    for a, b in combinations(kich_ban, 2):
        key = frozenset({a, b})

        if key in cap_xung_dot:                       # xung đột -> loại cả tổ hợp
            return None, f"{a} <-> {b}: {cap_xung_dot[key]}"

        ht, xd, diem = tinh_diem_cap_cache(a, b, cache, model, tf)

        ghi_chu = ""
        if key in cap_ho_tro:                         # hỗ trợ -> cộng điểm thưởng
            diem += DIEM_THUONG_HO_TRO
            ghi_chu = f"[HỖ TRỢ +{DIEM_THUONG_HO_TRO}: {cap_ho_tro[key]}]"

        tong += diem
        chi_tiet.append((a, b, ht, xd, diem, ghi_chu))
    return tong, chi_tiet


def tinh_top_to_hop(danh_sach_vien, cap_xung_dot, cap_ho_tro, model, tf,
                    top_k=3, toi_da_kich_ban=5000):
    """
    danh_sach_vien: list các viên, mỗi viên là list 1–3 tên thuốc.
      ["A"]           -> chắc chắn là A
      ["A","B","C"]   -> nghi ngờ là A hoặc B hoặc C
    Trả về top_k tổ hợp điểm cao nhất: list (điểm, tổ_hợp, chi_tiết).
    """
    print("\n" + "=" * 60)
    print(f"TOP {top_k} TỔ HỢP CÓ ĐIỂM CAO NHẤT")
    print("=" * 60)

    if len(danh_sach_vien) < 2:
        print("  Cần ít nhất 2 viên thuốc.")
        return None

    so_kb = 1
    for v in danh_sach_vien:
        so_kb *= len(v)
    if so_kb > toi_da_kich_ban:
        print(f"  [CẢNH BÁO] Quá nhiều kịch bản ({so_kb} > {toi_da_kich_ban}). "
              f"Hãy giảm bớt số viên nghi ngờ.")
        return None

    cache, ket_qua = {}, []
    so_trung_ten = so_xung_dot = 0
    for kb in product(*danh_sach_vien):
        if len(set(kb)) < len(kb):                    # 2 viên cùng 1 tên thuốc -> bỏ qua
            so_trung_ten += 1
            continue
        diem, chi_tiet = tinh_diem_kich_ban(kb, cache, cap_xung_dot, cap_ho_tro, model, tf)
        if diem is None:                              # có xung đột -> loại
            so_xung_dot += 1
            continue
        ket_qua.append((diem, kb, chi_tiet))

    print(f"  Tổng {so_kb} kịch bản | loại do xung đột: {so_xung_dot} | "
          f"loại do trùng tên: {so_trung_ten} | còn lại: {len(ket_qua)}\n")

    if not ket_qua:
        print("  Không còn tổ hợp nào hợp lệ (tất cả đều xung đột hoặc trùng tên).")
        return None

    ket_qua.sort(key=lambda x: x[0], reverse=True)
    top = ket_qua[:top_k]

    for hang, (diem, kb, ct) in enumerate(top, 1):
        print(f"  HẠNG {hang}: {' + '.join(kb)}")
        print(f"    Điểm tổng: {diem:+.4f}")
        for a, b, ht, xd, d, gc in ct:
            print(f"      {a} <-> {b}: hỗ trợ={ht:+.4f}, xung đột={xd:+.4f}, "
                  f"điểm cặp={d:+.4f} {gc}")
        print()
    print("=" * 60)
    return top


print("Đã nạp xong các hàm chấm điểm.")

Đã nạp xong các hàm chấm điểm.


## 6. Xây đồ thị + train PyKEEN (chạy 1 lần)

In [12]:
cap_xung_dot, cap_ho_tro, all_triples = xay_do_thi_va_lay_du_lieu()
pykeen_model, pykeen_tf = train_pykeen(all_triples)

GIAI ĐOẠN 1: XÂY ĐỒ THỊ HỖ TRỢ + XUNG ĐỘT TRONG NEO4J


INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [40, 9, 10]


  Đã nạp 50 cặp hỗ trợ, 45 cặp xung đột.
  Tổng 95 triple để train PyKEEN.

GIAI ĐOẠN 2: TRAIN PYKEEN (Knowledge Graph Embedding)


INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/1.00 [00:00<?, ?batch/s]

... (98 outputs left out)

Training batches on cuda:0:   0%|          | 0.00/1.00 [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/9.00 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.28s seconds


  Train xong! Loss cuối: 0.2818



## 7. Nhập thuốc và xem top 3 tổ hợp

- **Chắc chắn**: gõ 1 tên, ví dụ `Paracetamol`
- **Nghi ngờ (tối đa 3)**: gõ nhiều tên cách nhau bằng `/`, ví dụ `Aspirin / Ibuprofen / Naproxen`
- Enter trống để dừng.

Chạy lại cell này nhiều lần để thử danh sách khác, không cần train lại.

In [13]:
danh_sach_vien = []
print("Nhập từng VIÊN thuốc:")
print("  - Chắc chắn      : gõ 1 tên          (vd: Paracetamol)")
print("  - Nghi ngờ (≤ 3) : nhiều tên cách nhau bằng '/'  (vd: Aspirin / Ibuprofen / Naproxen)")
print("  - Enter trống để dừng.\n")

while True:
    s = input(f"Viên #{len(danh_sach_vien) + 1}: ").strip()
    if not s:
        break

    ung_vien = [x.strip() for x in re.split(r"[/,]", s) if x.strip()]
    ung_vien = list(dict.fromkeys(ung_vien))          # bỏ tên trùng, giữ thứ tự
    if not ung_vien:
        continue

    if len(ung_vien) > 3:
        print("  [!] Tối đa 3 tên cho 1 viên, chỉ lấy 3 tên đầu.")
        ung_vien = ung_vien[:3]

    if pykeen_tf is not None:
        la = [t for t in ung_vien if t not in pykeen_tf.entity_to_id]
        if la:
            print(f"  [!] Chưa có trong đồ thị (điểm sẽ = 0): {', '.join(la)}")

    danh_sach_vien.append(ung_vien)
    loai = "chắc chắn" if len(ung_vien) == 1 else f"nghi ngờ ({len(ung_vien)} khả năng)"
    print(f"  -> Viên #{len(danh_sach_vien)}: {' | '.join(ung_vien)}  [{loai}]")

if len(danh_sach_vien) >= 2:
    top3 = tinh_top_to_hop(danh_sach_vien, cap_xung_dot, cap_ho_tro,
                           pykeen_model, pykeen_tf,
                           top_k=TOP_K, toi_da_kich_ban=TOI_DA_KICH_BAN)
else:
    print("\nCần nhập ít nhất 2 viên thuốc.")

Nhập từng VIÊN thuốc:
  - Chắc chắn      : gõ 1 tên          (vd: Paracetamol)
  - Nghi ngờ (≤ 3) : nhiều tên cách nhau bằng '/'  (vd: Aspirin / Ibuprofen / Naproxen)
  - Enter trống để dừng.

Viên #1: Zinc
  -> Viên #1: Zinc  [chắc chắn]
Viên #2: Esomeprazol 20mg/Cefpobiotic 200/Mebilax 15
  -> Viên #2: Esomeprazol 20mg | Cefpobiotic 200 | Mebilax 15  [nghi ngờ (3 khả năng)]
Viên #3: Bar
  -> Viên #3: Bar  [chắc chắn]
Viên #4: Becacold-E
  -> Viên #4: Becacold-E  [chắc chắn]
Viên #5: Kremil-S/Cefpobiotic 200/Celecoxib
  -> Viên #5: Kremil-S | Cefpobiotic 200 | Celecoxib  [nghi ngờ (3 khả năng)]
Viên #6: 

TOP 3 TỔ HỢP CÓ ĐIỂM CAO NHẤT
  Tổng 9 kịch bản | loại do xung đột: 4 | loại do trùng tên: 1 | còn lại: 4

  HẠNG 1: Zinc + Esomeprazol 20mg + Bar + Becacold-E + Celecoxib
    Điểm tổng: +11.8150
      Zinc <-> Esomeprazol 20mg: hỗ trợ=-10.6882, xung đột=-9.1088, điểm cặp=-1.5794 
      Zinc <-> Bar: hỗ trợ=-9.9978, xung đột=-9.6790, điểm cặp=-0.3188 
      Zinc <-> Becacold-E: hỗ tr